# US Aviation — AI Explainability
### Permutation Feature Importance · Random Forest

Computes permutation-based feature importance on the held-out test set (20% of
`flights_model_ready_3class.csv`) using the trained Random Forest model.

Outputs:
| File | Contents |
|---|---|
| `reports/feature_importance.csv` | Permutation importance mean & std per feature |
| `figures/permutation_importance_top15.png` | Top-15 chart (raw feature names) |
| `figures/permutation_importance_top15_clean.png` | Top-15 chart (clean labels) |
| `reports/local_prediction_examples.csv` | 5 correct + 5 incorrect high-confidence predictions |
| `reports/explainability_report.json` | Full report with top-10 features |

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1 — GPU Check & Install cuML
cuML is required to load the Random Forest model saved by the training notebook.

In [2]:
import subprocess, sys

def _has_gpu():
    try:
        subprocess.run(["nvidia-smi"], check=True, capture_output=True)
        return True
    except Exception:
        return False

USE_GPU = _has_gpu()

if USE_GPU:
    print("GPU detected — installing cuML…")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet",
         "--extra-index-url", "https://pypi.nvidia.com",
         "cuml-cu12", "cudf-cu12"],
        check=True
    )
    print("cuML installed.")
else:
    print("No GPU — will load RF as sklearn model (CPU).")

GPU detected — installing cuML…
cuML installed.


In [3]:
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
print("Libraries loaded.")

Libraries loaded.


## Step 2 — Configure Paths

In [4]:

BASE        = Path("/content/drive/MyDrive/aviation")
MODEL_READY = BASE / "flights_model_ready_3class.csv"
MODELS_DIR  = BASE / "models"
REPORTS_DIR = BASE / "reports"
FIGURES_DIR = BASE / "figures"


REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL   = "delay_class_3"
RANDOM_STATE = 42

PERM_CSV              = REPORTS_DIR / "feature_importance.csv"
EXPLAIN_REPORT        = REPORTS_DIR / "explainability_report.json"
PERM_FIG              = FIGURES_DIR / "permutation_importance_top15.png"
PERM_FIG_CLEAN        = FIGURES_DIR / "permutation_importance_top15_clean.png"
LOCAL_EXPLANATION_CSV = REPORTS_DIR / "local_prediction_examples.csv"

print("Paths configured.")
print(f"  MODEL_READY : {MODEL_READY}")
print(f"  MODELS_DIR  : {MODELS_DIR}")

Paths configured.
  MODEL_READY : /content/drive/MyDrive/aviation/flights_model_ready_3class.csv
  MODELS_DIR  : /content/drive/MyDrive/aviation/models


## Step 3 — Load Data & Split

Same 80/20 stratified split as training (random_state=42) to get the identical test set.

In [5]:
print("Loading model-ready dataset…")
df = pd.read_csv(MODEL_READY, low_memory=False)
print(f"Shape: {df.shape}")

DROP_COLS = ["FL_DATE", "AIRLINE_CODE", "route"]
df_feat   = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")

X_raw = df_feat.drop(columns=[TARGET_COL])
y     = df_feat[TARGET_COL].astype(int).values

cat_cols = X_raw.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = X_raw.select_dtypes(include=["number"]).columns.tolist()

_, X_test_raw, _, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Test set : {X_test_raw.shape[0]:,} rows  ×  {X_test_raw.shape[1]} features")
print(f"Numeric cols   : {num_cols}")
print(f"Categorical cols: {cat_cols}")

Loading model-ready dataset…
Shape: (2912112, 23)
Test set : 582,423 rows  ×  19 features
Numeric cols   : ['CRS_DEP_TIME_MIN', 'CRS_ARR_TIME_MIN', 'CRS_ELAPSED_TIME', 'DISTANCE', 'SCHED_BLOCK_MINS', 'year', 'month', 'day', 'day_of_week', 'is_weekend', 'dep_hour', 'arr_hour']
Categorical cols: ['AIRLINE', 'ORIGIN', 'DEST', 'ORIGIN_STATE', 'DEST_STATE', 'dep_time_bucket', 'season']


## Step 4 — Load Model & Preprocessors

In [6]:
print("Loading Random Forest model…")
rf_model = joblib.load(MODELS_DIR / "random_forest_3class.joblib")
print(f"Loaded: {type(rf_model).__name__}")

print("\nLoading preprocessors…")
ord_enc = joblib.load(MODELS_DIR / "ordinal_encoder.joblib")
scaler  = joblib.load(MODELS_DIR / "standard_scaler.joblib")
print("OrdinalEncoder and StandardScaler loaded.")

Loading Random Forest model…
Loaded: RandomForestClassifier

Loading preprocessors…
OrdinalEncoder and StandardScaler loaded.


## Step 5 — Preprocess Test Set

Apply the same OrdinalEncoder + StandardScaler fitted during training
to get the float32 array the model expects.

In [7]:
X_cat = ord_enc.transform(X_test_raw[cat_cols]).astype("float32")
X_num = scaler.transform(X_test_raw[num_cols]).astype("float32")
X_test = np.hstack([X_num, X_cat])

feature_names = num_cols + cat_cols

print(f"Preprocessed test set: {X_test.shape}  dtype={X_test.dtype}")

Preprocessed test set: (582423, 19)  dtype=float32


## Step 6 — Permutation Feature Importance

Shuffles each feature column and measures the drop in weighted F1.
`n_repeats=3` runs each shuffle 3 times and averages; this takes a few minutes.

In [8]:
print("Running permutation importance (n_repeats=3, scoring=f1_weighted)…")
print("This may take several minutes on the full test set.")

perm = permutation_importance(
    rf_model,
    X_test,
    y_test,
    n_repeats=3,
    random_state=RANDOM_STATE,
    scoring="f1_weighted",
    n_jobs=-1,
)

print("Permutation importance complete.")

Running permutation importance (n_repeats=3, scoring=f1_weighted)…
This may take several minutes on the full test set.
Permutation importance complete.


## Step 7 — Build Results DataFrame & Save CSV

In [9]:
def clean_feature_name(name: str) -> str:
    replacements = {
        "CRS_DEP_TIME_MIN": "Scheduled Departure Time",
        "CRS_ARR_TIME_MIN": "Scheduled Arrival Time",
        "CRS_ELAPSED_TIME": "Scheduled Elapsed Time",
        "SCHED_BLOCK_MINS": "Scheduled Block Time",
        "DISTANCE": "Distance",
        "FL_DATE": "Flight Date",
        "AIRLINE": "Airline",
        "AIRLINE_CODE": "Airline Code",
        "ORIGIN": "Origin Airport",
        "DEST": "Destination Airport",
        "ORIGIN_STATE": "Origin State",
        "DEST_STATE": "Destination State",
        "dep_hour": "Departure Hour",
        "arr_hour": "Arrival Hour",
        "day_of_week": "Day of Week",
        "is_weekend": "Weekend Flag",
        "month": "Month",
        "day": "Day",
        "year": "Year",
        "route": "Route",
        "season": "Season",
        "dep_time_bucket": "Departure Time Bucket",
    }
    return replacements.get(name, name.replace("_", " ").title())


perm_df = pd.DataFrame({
    "feature": feature_names,
    "clean_feature": [clean_feature_name(c) for c in feature_names],
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

perm_df.to_csv(PERM_CSV, index=False)
print(f"Saved → {PERM_CSV}")
print(f"\nTop 10 features:")
print(perm_df[["feature", "importance_mean", "importance_std"]].head(10).to_string(index=False))

Saved → /content/drive/MyDrive/aviation/reports/feature_importance.csv

Top 10 features:
     feature  importance_mean  importance_std
     AIRLINE         0.014973        0.000085
        year         0.013105        0.000397
         day         0.002246        0.000095
       month         0.001635        0.000178
 day_of_week         0.001170        0.000061
  is_weekend        -0.000017        0.000055
      ORIGIN        -0.000226        0.000005
      season        -0.000318        0.000104
ORIGIN_STATE        -0.000692        0.000034
        DEST        -0.000747        0.000054


## Step 8 — Plot Feature Importance

In [10]:
top15 = perm_df.head(15).sort_values("importance_mean", ascending=True)

# Raw feature names
plt.figure(figsize=(10, 7))
plt.barh(top15["feature"], top15["importance_mean"], xerr=top15["importance_std"])
plt.title("Permutation Feature Importance - Top 15")
plt.xlabel("Decrease in Weighted F1 After Shuffling")
plt.tight_layout()
plt.savefig(PERM_FIG, dpi=200, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved → {PERM_FIG}")

# Clean feature names
plt.figure(figsize=(10, 7))
plt.barh(top15["clean_feature"], top15["importance_mean"], xerr=top15["importance_std"])
plt.title("Permutation Feature Importance - Top 15")
plt.xlabel("Decrease in Weighted F1 After Shuffling")
plt.tight_layout()
plt.savefig(PERM_FIG_CLEAN, dpi=200, bbox_inches="tight")
plt.show()
plt.close()
print(f"Saved → {PERM_FIG_CLEAN}")

Saved → /content/drive/MyDrive/aviation/figures/permutation_importance_top15.png
Saved → /content/drive/MyDrive/aviation/figures/permutation_importance_top15_clean.png


## Step 9 — Local Prediction Examples

Top-5 correct predictions (highest confidence) and
top-5 incorrect predictions (highest confidence) from the test set.

In [11]:
if USE_GPU:
    import cupy as cp
    X_in  = cp.array(X_test)
    probs = rf_model.predict_proba(X_in)
    preds = rf_model.predict(X_in)
    probs = probs.get() if hasattr(probs, "get") else np.array(probs)
    preds = preds.get().astype(int) if hasattr(preds, "get") else np.array(preds).astype(int)
else:
    probs = rf_model.predict_proba(X_test)
    preds = rf_model.predict(X_test).astype(int)

local_df = X_test_raw.copy().reset_index(drop=True)
local_df["actual_class"]          = y_test
local_df["predicted_class"]       = preds
local_df["prob_class_0"]          = probs[:, 0]
local_df["prob_class_1"]          = probs[:, 1]
local_df["prob_class_2"]          = probs[:, 2]
local_df["prediction_confidence"] = local_df[["prob_class_0", "prob_class_1", "prob_class_2"]].max(axis=1)

examples = pd.concat([
    local_df[local_df["actual_class"] == local_df["predicted_class"]]
    .sort_values("prediction_confidence", ascending=False)
    .head(5),
    local_df[local_df["actual_class"] != local_df["predicted_class"]]
    .sort_values("prediction_confidence", ascending=False)
    .head(5),
], ignore_index=True)

examples.to_csv(LOCAL_EXPLANATION_CSV, index=False)
print(f"Saved → {LOCAL_EXPLANATION_CSV}")
examples[["actual_class", "predicted_class", "prediction_confidence",
          "prob_class_0", "prob_class_1", "prob_class_2"]].head(10)

Saved → /content/drive/MyDrive/aviation/reports/local_prediction_examples.csv


,actual_class,predicted_class,prediction_confidence,prob_class_0,prob_class_1,prob_class_2
0,0,0,0.958182,0.958182,0.024320,0.017498
1,0,0,0.947061,0.947061,0.027910,0.025029
2,0,0,0.945279,0.945279,0.026356,0.028365
3,0,0,0.945107,0.945107,0.028332,0.026561
4,0,0,0.942616,0.942616,0.042961,0.014423
5,1,0,0.914899,0.914899,0.054253,0.030848
6,1,0,0.908861,0.908861,0.058476,0.032663
7,2,0,0.904330,0.904330,0.061663,0.034007
8,1,0,0.903440,0.903440,0.057946,0.038614
9,2,0,0.903152,0.903152,0.041021,0.055827


## Step 10 — Save Explainability Report

In [12]:
report = {
    "model_used": "random_forest_3class",
    "input_data": str(MODEL_READY),
    "method": "Permutation Feature Importance",
    "metric_used": "weighted_f1",
    "permutation_importance_csv": str(PERM_CSV),
    "permutation_importance_figure": str(PERM_FIG),
    "permutation_importance_clean_figure": str(PERM_FIG_CLEAN),
    "local_prediction_examples_csv": str(LOCAL_EXPLANATION_CSV),
    "top_10_features": perm_df.head(10).to_dict(orient="records"),
}

EXPLAIN_REPORT.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("Saved permutation importance CSV to:", PERM_CSV)
print("Saved permutation importance figure to:", PERM_FIG)
print("Saved clean permutation importance figure to:", PERM_FIG_CLEAN)
print("Saved local prediction examples to:", LOCAL_EXPLANATION_CSV)
print("Saved explainability report to:", EXPLAIN_REPORT)

Saved permutation importance CSV to: /content/drive/MyDrive/aviation/reports/feature_importance.csv
Saved permutation importance figure to: /content/drive/MyDrive/aviation/figures/permutation_importance_top15.png
Saved clean permutation importance figure to: /content/drive/MyDrive/aviation/figures/permutation_importance_top15_clean.png
Saved local prediction examples to: /content/drive/MyDrive/aviation/reports/local_prediction_examples.csv
Saved explainability report to: /content/drive/MyDrive/aviation/reports/explainability_report.json


## Done

All outputs saved to Google Drive:

| File | Contents |
|---|---|
| `reports/feature_importance.csv` | Permutation importance mean & std per feature |
| `reports/local_prediction_examples.csv` | 5 correct + 5 incorrect high-confidence examples |
| `reports/explainability_report.json` | Full report with top-10 features |
| `figures/permutation_importance_top15.png` | Top-15 chart (raw feature names) |
| `figures/permutation_importance_top15_clean.png` | Top-15 chart (clean labels) |